# Scratch fast → stable large: fair full-test comparison

Run **fast first, then large** in separate NEW v4 Drive runs. Select T4 GPU. Fast retains its successful Post-LN architecture/init/Noam settings as a control; large switches to Pre-LN, scaled embeddings, tied output, longer warmup and lower peak LR. Both use schema v2, identical test references and full-test beam 4 + greedy. Large additionally records trigram suppression off/on. The old collapsed run must not be resumed. Helsinki is already trained separately; point the last cell to its existing results. GPU learning is performed by the user.

## 1. Clone/update and live stdout/stderr streaming
A clean checkout is fast-forwarded from `feat/scratch-transformer`; local changes are preserved by failing early. Child failures propagate, including divergence exit code 2. After merge the ref may be changed to `main`.

In [ ]:
from pathlib import Path
import subprocess
import sys

def stream(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)

REPO_URL = "https://github.com/vcdx1111/Machine-translation-English-to-Russian-.git"
REPO_REF = "feat/scratch-transformer"
REPO = Path("/content/Machine-translation-English-to-Russian-")
if not REPO.exists():
    stream(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO)])
else:
    assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Preserve local changes before updating"
    stream(["git", "-C", str(REPO), "fetch", "origin", REPO_REF])
    stream(["git", "-C", str(REPO), "switch", REPO_REF])
    stream(["git", "-C", str(REPO), "merge", "--ff-only", "origin/" + REPO_REF])
%cd /content/Machine-translation-English-to-Russian-
stream(["git", "log", "-1", "--oneline"])

## 2. Install and select CUDA
Install before imports; restart the runtime and rerun if Colab requests it. FP16 status and first-batch logits dtype are checked by training. This notebook does not retrain Helsinki.

In [ ]:
stream([sys.executable,"-m","pip","install","-r","requirements.txt"])
import torch
assert torch.cuda.is_available(), "Select a T4 GPU runtime"
print(torch.__version__,torch.cuda.get_device_name(0))

## 3. Drive and experiment names
Use these NEW v4 names. After disconnect set the unfinished run's `resume=True`. Completed/pruned runs are skipped when rerunning the notebook, after checking their config. Never run two writers to one run. Keep metrics/logs/samples even after checkpoint cleanup. The cleanup cells remove resume capability deliberately, only after successful train AND evaluation. A diverged run is preserved for diagnosis.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import json, yaml, hashlib
ROOT = Path("/content/drive/MyDrive/en_ru_translation")
FAST = dict(config="configs/transformer_colab_fast.yaml",name="scratch_fast_v4_fullbeam",resume=False)
LARGE = dict(config="configs/transformer_colab_large.yaml",name="scratch_large_stable_v4",resume=False)
for spec in (FAST,LARGE):
    spec["checkpoints"] = ROOT/spec["name"]/"checkpoints"
    spec["results"] = ROOT/spec["name"]/"results"
    spec["checkpoints"].mkdir(parents=True,exist_ok=True)
    spec["results"].mkdir(parents=True,exist_ok=True)

def completed(spec):
    path=spec["results"]/"run_status.json"
    if not path.exists(): return False
    status=json.loads(path.read_text())
    if status["status"]=="diverged":
        raise RuntimeError("Diverged run: inspect divergence.json/step_log.csv and choose a NEW run name")
    done=status.get("evaluation_complete") and status["status"] in ("completed","early_stopped","pruned")
    if done:
        record=json.loads((spec["results"]/"metrics.json").read_text())
        actual=record["config"]
        expected=yaml.safe_load(Path(spec["config"]).read_text())
        assert record["data_sha256"]==hashlib.sha256(Path(expected["paths"]["raw"]).read_bytes()).hexdigest(), "Raw data changed; use a NEW run name"
        for key in ("seed","model","training","tokenizer","data"):
            assert actual[key]==expected[key], "Completed config changed; choose a NEW run name"
    return done

def command(spec):
    return [sys.executable,"-u","-m","src.train","--config",spec["config"],
            "--checkpoint-dir",str(spec["checkpoints"]),"--results-dir",str(spec["results"])]

def measure(spec):
    if completed(spec):
        print(spec["name"],"already evaluated; skipping")
    elif not spec["resume"]:
        assert not (spec["checkpoints"]/"last.pt").exists(), "Set resume=True or use a new run name"
        stream(command(spec)+["--stop-after-epochs","1","--skip-test"])
    else:
        assert (spec["checkpoints"]/"last.pt").exists(), "No v4 resume checkpoint"

def show_estimate(spec):
    path=spec["results"]/"runtime_estimate.json"
    if path.exists(): print(path.read_text())

def finish(spec):
    if not completed(spec):
        stream(command(spec)+["--resume",str(spec["checkpoints"]/"last.pt")])
    print((spec["results"]/"metrics.json").read_text())

## 4. Fast: measure one epoch
Fast keeps 256/3, legacy Xavier embeddings, sqrt(d_model) scaling, untied output, Post-LN, warmup 800. Only logging/guards and full beam-4 evaluation change. The original user run took 447 seconds for 12 epochs; plan roughly **10–20 minutes** including new full evaluation/logging/Drive I/O, not a guarantee. Both models retain the same grouped holdout; tokenizers and caps differ, so this is a system comparison, not a one-variable capacity ablation.

In [ ]:
measure(FAST)
show_estimate(FAST)

## 5. Fast: continue the same checkpoint and evaluate
Logs stream step progress/ETA; `step_log.csv` records every attempt's unscaled gradient norm, clipped norm, proposed LR, actual applied LR, optimizer step, scaler before/after and skip status. NaN/inf loss or sharp validation regression triggers a warning and stops. A few initial FP16 skipped steps can be scaler calibration; repeated overflows are guarded. Final metrics always distinguish full beam and full greedy.

In [ ]:
finish(FAST)
print((FAST["results"]/"train_log.csv").read_text())

## 6. Fast: keep ONLY compact best.pt
Run AFTER the successful evaluation above. This verifies run IDs/results and atomically compacts `best.pt`, removes optimizer/scaler/RNG/current weights and deletes `last.pt`. It retains the tokenizer and all result/log files. The remaining best checkpoint supports evaluation/translation **but cannot resume training**. Running/calibrated/diverged runs are refused; repeat cleanup is safe.

In [ ]:
stream([sys.executable,"-m","src.checkpoints","--checkpoint-dir",str(FAST["checkpoints"]),
        "--results-dir",str(FAST["results"])])

## 7. Stable large: measure one epoch
Large now uses Pre-LN plus final encoder/decoder norms; d_model 512, six layers, 16k BPE, cap 64; scaled embedding initialization and tied output. Warmup **3000**, factor **.4**, peak LR **~3.23e-4**; padded token budget **16,384**. Thirty epochs, patience five. The old unstable model measured 137 s/epoch at 32k tokens; initially allow **~75–150 minutes** for this new run. More updates, Pre-LN, lower LR and longer training change that estimate. The first epoch/projection replaces it. Toy CPU overfit passes with the full 512×6/16k model; this does not establish T4 convergence.

In [ ]:
measure(LARGE)
show_estimate(LARGE)

## 8. Stable large: continue and evaluate
The scalar divergence guard stops when val_loss is both >1.25× best and ≥0.5 above best; no failed weights replace the best checkpoint. Nonfinite loss stops immediately. Persistent overflow stops training. A stopped run records status/reason and preserves diagnostics; the CLI exits nonzero on divergence even if it can evaluate older best weights. Full beam-4 suppression off remains the primary result; greedy and trigram-suppression on are recorded separately.

In [ ]:
finish(LARGE)
print((LARGE["results"]/"train_log.csv").read_text())

## 9. Stable large: keep ONLY compact best.pt
Same completion/matching checks as fast. This intentionally removes training resume capability to save Drive space; it does not delete results or unrelated files. To train longer later, keep the full checkpoint instead of running this cell.

In [ ]:
stream([sys.executable,"-m","src.checkpoints","--checkpoint-dir",str(LARGE["checkpoints"]),
        "--results-dir",str(LARGE["results"])])

## 10. Four-way comparison with existing Helsinki results
Set the path to the completed Helsinki result root containing `zero_shot/` and `finetuned/`. This cell does not train/download Helsinki. The script requires all four runs, rejects known reference/signature mismatches, labels unreported status/legacy identity, and never mixes subset scores into full-test scores. Tatoeba may have been in OPUS pretraining; high BLEU does not rule out unsupported tails/hallucinations. Review examples for source faithfulness.

In [ ]:
HELSINKI_RESULTS = ROOT/"helsinki_v1_seed42"/"results"
stream([sys.executable,"scripts/compare_results.py",str(FAST["results"]),str(LARGE["results"]),
        str(HELSINKI_RESULTS),"--require-four-models","--output",str(ROOT/"comparison_v4")])

## 11. Send back
Send metrics, train_log.csv, step_log.csv, runtime_estimate.json, run_status.json, samples (large off/on), divergence.json if produced, exact config/commit, GPU memory/epoch time and errors. Helsinki's reported extra unsupported tail is a faithfulness limitation, not something trigram suppression reliably fixes. Keep compact best.pt on Drive for inference; do not commit weights or secrets.